In [2]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 170.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 6.0 MB/s eta 0:00:0000:010:010m

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Stanford

In [7]:
import pandas as pd

stanford = pd.read_csv("./stanford.csv", index_col=0)
stanford.head()

,class,A+,A,A-,B+,B,B-,C+,C,C-,D+,D,D-,F,CR,NC
0,BIOE191,0.099467,0.763766,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.017762,0.119005,0.0
1,SPANLANG12C,0.230769,0.393773,0.216117,0.100733,0.028388,0.000000,0.002747,0.0,0.0,0.0,0.0,0.0,0.006410,0.021062,0.0
2,ENGLISH94,0.137476,0.809793,0.026365,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.026365,0.0
3,POLISCI236,0.090268,0.303244,0.272920,0.151622,0.181946,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
4,AMSTUD160,0.016374,0.502924,0.242105,0.138012,0.051462,0.016374,0.000000,0.0,0.0,0.0,0.0,0.0,0.016374,0.016374,0.0


In [8]:
# Drop CR and NC columns
grade_cols = [c for c in stanford.columns if c not in ['class', 'CR', 'NC']]
df_grades = stanford[['class'] + grade_cols].copy()

# Filter out classes where all grade buckets are zero (meaning they only had CR/NC)
df_grades = df_grades[df_grades[grade_cols].sum(axis=1) > 0]

# Normalize the distribution among the remaining buckets
df_normalized = df_grades.copy()
df_normalized[grade_cols] = df_normalized[grade_cols].div(df_normalized[grade_cols].sum(axis=1), axis=0)

df_normalized.head()

,class,A+,A,A-,B+,B,B-,C+,C,C-,D+,D,D-,F
0,BIOE191,0.112903,0.866935,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.020161
1,SPANLANG12C,0.235734,0.402245,0.220767,0.102900,0.028999,0.000000,0.002806,0.0,0.0,0.0,0.0,0.0,0.006548
2,ENGLISH94,0.141199,0.831721,0.027079,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
3,POLISCI236,0.090268,0.303244,0.272920,0.151622,0.181946,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
4,AMSTUD160,0.016647,0.511296,0.246136,0.140309,0.052319,0.016647,0.000000,0.0,0.0,0.0,0.0,0.0,0.016647


In [14]:
import json
from datetime import datetime

def save_grade_lookup(df, school_name, filename, grade_columns):
    """Helper to save normalized grade distribution to JSON."""
    lookup = {
        "school": school_name,
        "last_update": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "classes": df.set_index('class')[grade_columns].to_dict(orient='index')
    }
    with open(f"out/{filename}", 'w') as f:
        json.dump(lookup, f, indent=2)
    print(f"Saved lookup for {len(lookup['classes'])} classes to {filename}")

# Save Stanford using the helper
save_grade_lookup(df_normalized, "Stanford", "stanford_grades.json", grade_cols)

Saved lookup for 696 classes to stanford_grades.json


## UT Austin

In [ ]:
ut_df = pd.read_csv(
    "./data/UT_austin.csv", 
    encoding='utf-16', 
    sep='\t', 
    engine='python', 
    on_bad_lines='warn'
)

# Create courseCode by combining Prefix and Number
ut_df['class'] = ut_df['Course Prefix'].str.strip() + ut_df['Course Number'].str.strip()

# Pivot to get grades as columns, summing the counts
ut_pivot = ut_df.pivot_table(
    index='class', 
    columns='Letter Grade', 
    values='Count of letter grade', 
    aggfunc='sum', 
    fill_value=0
).reset_index()

# Standard grade buckets we care about
target_grades = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D', 'D-', 'F']

# Ensure all target columns exist
for g in target_grades:
    if g not in ut_pivot.columns:
        ut_pivot[g] = 0

# Filter out classes with no grades in our target buckets
ut_pivot = ut_pivot[ut_pivot[target_grades].sum(axis=1) > 0]

# Normalize
ut_normalized = ut_pivot[['class'] + target_grades].copy()
ut_normalized[target_grades] = ut_normalized[target_grades].div(ut_normalized[target_grades].sum(axis=1), axis=0)

# Save using helper
save_grade_lookup(ut_normalized, "UT Austin", "ut_austin_grades.json", target_grades)
ut_normalized.head()

Saved lookup for 3289 classes to ut_austin_grades.json


Letter Grade,class,A+,A,A-,B+,B,B-,C+,C,C-,D+,D,D-,F
0,A I388,0.0,0.351695,0.309322,0.220339,0.055085,0.025424,0.021186,0.004237,0.000000,0.000000,0.0,0.0,0.012712
1,A I388J,0.0,0.261538,0.207692,0.269231,0.192308,0.015385,0.023077,0.000000,0.000000,0.000000,0.0,0.0,0.030769
2,A I388K,0.0,0.721854,0.145695,0.119205,0.006623,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.006623
3,A I388U,0.0,0.597403,0.136364,0.038961,0.064935,0.077922,0.025974,0.000000,0.012987,0.006494,0.0,0.0,0.038961
4,A I389L,0.0,0.233645,0.158879,0.523364,0.037383,0.037383,0.000000,0.009346,0.000000,0.000000,0.0,0.0,0.000000
